# Lab 4: Build a Pipeline for New Data

As you have been working with the PFL dataset, you have noticed that there is something missing. American Football is a risky sport. Injuries are common and can often affect the outcome of future games. It would be very useful for a sports analytics firm to track data on injuries.

**Objectives:** In this lab exercise, you will use Snowflake CoCo to create a new table to store data on injured players and create a pipeline that will ingest updates to this data.

## Setup

First, let's get your environment set up. Run the cell below to generate the PFL dataset - a database of information about the fictional Professional Football League.

In [ ]:
from resources.setup_pfl import setup_pfl
setup_pfl()

The database `PFL_DB` has been created with the schema `STATS_AND_INFO` that holds all of the tables and views relating to the Professional Football League.

Run the cell below to set your context to this database and schema.

In [ ]:
%%sql -r Set_Context
CREATE WAREHOUSE IF NOT EXISTS COCOLABS_WH WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE COCOLABS_WH;
USE DATABASE PFL_DB;
USE SCHEMA STATS_AND_INFO;

## Create a New Table

You know that you will have to create a new table in the database to store information about injuries. But you are not quite sure how to create this table, what types of information it will hold, and how it will relate to other tables in the database.

### Ask Snowflake CoCo for advice

Fortunately, Snowflake CoCo can help plan the table for our new data. Let's see what advice it can give.

In the Snowflake CoCo side-panel, enter the following prompt:

<code style="display:block; user-select:all; padding: 15px;">
I want to add a table to store information about injured players in the @PFL_DB database. What columns should it have?
</code>

Review the recommendation. The results might vary slightly every time you ask. The recommendation should have, at the least, the following characteristics:

- An ID column for the injury
- Columns that reference the PLAYER_ID, TEAM_ID, and SEASON_ID, that are used as foreign keys.
- Details about the injury, such as injury type and body part
- Dates showing when the injury occurred and when the player was able to return

If the recommendation is missing any of these columns, ask Snowflake CoCo to update its recommendation. For example:

<code style="display:block; user-select:all; padding: 15px;">
Update the recommendation with a column for the date of the injury.
</code>

### Plan the table

Now that you have an idea of what the table will look like, you can create it. Snowflake CoCo is fully capable of creating Snowflake objects such as databases, schemas, or tables. However, it is often helpful to confirm what CoCo's plan is before actually making changes to your system.

At the bottom of the Snowflake CoCo side-panel, turn on the **Plan** switch to enter Plan Mode.

<div style="max-width: 450px;">

![Plan mode switch toggled on](resources/img/plan-mode-on.png)</div>

Then, enter the following prompt:

<code style="display:block; user-select:all; padding: 15px;">
Create the table according to your recommendation and name the table INJURIES.
</code>

Snowflake CoCo will show you its plan, step-by-step.

### Execute the plan

If the plan looks acceptable, enter the following prompt:

<code style="display:block; user-select:all; padding: 15px;">
Proceed.
</code>

You will occasionally be asked to Allow or Skip SQL statements or code that makes modifications. Even though you previously confirmed the plan, CoCo will not make changes to your environment without your approval.

Review any code and click **Allow** at these prompts.

<div style="max-width: 450px;">

![Clicking Allow](resources/img/allow-table-create.png)</div>

Once complete, confirm that the table exists by running the SQL cell below.

In [ ]:
%%sql -r Confirm_Injuries_Table
DESCRIBE TABLE PFL_DB.STATS_AND_INFO.INJURIES;

## Create Sample Data

Now that you have a table to store injuries, you would like to test it to see how well it works with the rest of the database. Unfortunately, the INJURIES table is empty and you do not yet have any real data for it. You need some temporary sample data for this table.

### Populate the table

Instead of inventing and typing out rows and rows of sample data, ask Snowflake CoCo to do it for you. Use the following prompt:

<code style="display:block; user-select:all; padding: 15px;">
Add 50 rows of realistic sample data for the @INJURIES table.
</code>

Since the INJURIES table has foreign keys to a number of other tables, you will notice that Snowflake CoCo has to gather information from those before proceeding. You will also be asked to **Allow** an insert statement.

When Coco finishes, run the SQL cell below to see the results.

In [ ]:
%%sql -r dataframe_2
SELECT * FROM PFL_DB.STATS_AND_INFO.INJURIES;

### Query the data

Test the data by running a query against it. Enter the following prompt using CoCo:

<code style="display:block; user-select:all; padding: 15px;">
Which players had leg injuries in the 2024 season?
</code>

## Import Real Data

You have now received some real data about injuries in a CSV file format. You expect that this is only the first of many such files. It would be nice to set up a pipeline that will automatically load these files as they come in.

### Recreate the injuries table

The injuries table that you created might not have the same column format as the CSV files. To simplify this lab and ensure that the table will accept data from the CSV files, you will recreate the INJURIES table. 

Run the code in the cell below.

In [ ]:
%%sql -r Recreate_injuries_table
DROP TABLE IF EXISTS PFL_DB.STATS_AND_INFO.INJURIES;

CREATE TABLE PFL_DB.STATS_AND_INFO.INJURIES (
    INJURY_ID VARCHAR,
    PLAYER_ID VARCHAR,
    TEAM_ID VARCHAR,
    SEASON_ID NUMBER,
    INJURY_TYPE VARCHAR,
    BODY_PART VARCHAR,
    INJURY_DATE DATE,
    EXPECTED_RETURN_DATE DATE,
    ACTUAL_RETURN_DATE DATE,
    GAMES_MISSED NUMBER,
    STATUS VARCHAR,
    IS_SEASON_ENDING BOOLEAN,
    NOTES VARCHAR
);

### Start a new CoCo chat

CoCo may still remember the previous INJURIES table, which may have had a different schema. To make sure there are no mixups, start a new CoCo chat by clicking the **New chat** button ( ![](resources/img/newchatbutton.png) ) at the top of the Coco panel.

### Create a stage for CSV files

The CSV files will be uploaded into a Snowflake stage. Run the cell below to create the stage.

In [ ]:
%%sql -r dataframe_4
CREATE OR REPLACE STAGE PFL_DB.STATS_AND_INFO.INJURIES_STAGE
 DIRECTORY = (ENABLE = TRUE);

## Planning the pipeline

You are now ready to create a simple data pipeline that will automatically load CSV files in from INJURIES_STAGE into the INJURIES table. Doing so requires a few new Snowflake objects to be created. 

Since this will be a bit more complex, turn on **Plan mode**.

<div style="max-width: 450px;">

![Plan mode switch toggled on](resources/img/plan-mode-on.png)</div>

Enter the following prompt into the Snowflake CoCo panel:

<code style="display:block; user-select:all; padding: 15px;">
Create a pipeline that loads data from CSV files located in the existing @PFL_DB.STATS_AND_INFO.INJURIES_STAGE into the @PFL_DB.STATS_AND_INFO.INJURIES table. Use 'Lab Materials/injuries_001.csv.gz' as a reference. The pipeline should check for new files every minute.
</code>

The results may vary, but will the plan will probably include:

- Ensuring that the stage has a directory table
- Creating a stream on the stage's directory table
- Creating a task that performs a COPY INTO command

If CoCo plans to test the pipeline by uploading a file, tell it to modify the plan to not include the test.


### Create the pipeline

Review the plan. If the process makes sense, confirm it by entering:

<code style="display:block; user-select:all; padding: 15px;">
Proceed
</code>

Allow any SQL statements as needed.

### Copy one CSV file into the stage

The pipeline that CoCo created checks the stage for new files every minute. Let's add a new CSV file to the stage.

On the left panel, find the **injuries_001.csv.gz** file, located in **CoCo Labs/Lab Materials**. 

Click the ellipsis menu next to it and select **Download**.

<div style="max-width: 400px;">

![](resources/img/download_injuries1.png)</div>

From the left menu in Snowflake, open **Catalog > Database Explorer**. Then browse to **PFL_DB > STATS_AND_INFO > Stages > INJURIES_STAGE**.

Click **+ Files**. Select the downloaded CSV file and upload it.

<div style="max-width: 750px;">

![](resources/img/add_files_injury_stage.png)</div>

### Confirm the pipeline is working

Wait one minute, then run the SQL cell below to see how many rows of data are in the INJURIES table. The first CSV file has 100 entries, so you should see 100 rows.

If you do not see any results, tell CoCo that pipeline is not working. Although CoCo does occasionally make mistakes, it is also very good ad fixing them.

In [ ]:
%%sql -r Confirm_pipeline
SELECT COUNT(1) FROM PFL_DB.STATS_AND_INFO.INJURIES;

### Add another file and confirm

From the panel on the left, download **injuries_002.csv.gz** and upload it into **INJURIES_STAGE** as you did before.

Wait one minute and confirm that the new data was added for a total of 200 rows.

In [ ]:
%%sql -r Confirm_pipeline_2
SELECT COUNT(1) FROM PFL_DB.STATS_AND_INFO.INJURIES;

### Stop the automated task

The task that  was created by this pipeline will continue to run indefinitely. This may slowly use your credits so it is best to suspend it.

Look at the output that CoCo generated to find the name of the task that it created. For example:

<code style="display:block; padding: 15px;">
CREATE OR REPLACE TASK <span style="font-weight:bold;color:blue;">PFL_DB.STATS_AND_INFO.LOAD_INJURIES_TASK</span> <br>
  SCHEDULE = '1 MINUTE'<br>
  ...
</code>

Insert the name of your task below and run the cell.

In [ ]:
ALTER TASK PFL_DB.STATS_AND_INFO.LOAD_INJURIES_TASK SUSPEND;

## Key Takeaways

In this lab, you explored how Snowflake CoCo can be used to:

- Create Snowflake objects such as tables, streams, and tasks
- Review complex procedures before executing them using Plan Mode
- Create sample data
- Create a simple data pipeline